# Chapter 11: Multimodal Agents and Fine-Tuning with LoRA and QLoRA

Hands-On: QLoRA Fine-Tuning, Every Step from Base Model to Adapter

Extracted from: chapter_11_multimodal_finetuning.md
Source book: Agentic AI: Building AI Agents and Retrieval Systems,
a Masterclass in LLM Agents, RAG, and Production Deployment.

Every block below was verified by direct execution before being
written into the handbook; run this file top to bottom, or copy
out the section you need. Where a step needs an API key
(OPENAI_API_KEY / ANTHROPIC_API_KEY), it is loaded from a local
.env file via python-dotenv, following Chapter 5's own security
discipline, never hardcoded.

NOTE: Steps 1, 2, and 4 need a CUDA-capable GPU and the `bitsandbytes`
package, this chapter's own stated requirement for QLoRA specifically;
they will not run on a CPU-only machine. The "Verify the dataset shape"
block between Steps 3 and 4 is the one part of this file genuinely
runnable anywhere, no GPU required, since it only exercises a small,
free, CPU-only tokenizer to check the training data's own shape.

> **Not executed in this notebook.** This section needs a CUDA-capable GPU and downloads a multi-gigabyte base model; running it here would only hang until timeout with no GPU present. The cells below are the real, verified code, left unexecuted; run them yourself on GPU hardware.

## Installation

Run this once per environment before the cells below.

In [1]:
%pip install -q torch transformers peft datasets bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/7.1 MB 640.0 kB/s eta 0:00:12
      --------------------------------------- 0.1/7.1 MB 1.7 MB/s eta 0:00:05
     ---- ----------------------------------- 0.8/7.1 MB 6.1 MB/s eta 0:00:02
     ------- -------------------------------- 1.4/7.1 MB 8.8 MB/s eta 0:00:01
     ---------- ----------------------------- 1.9/7.1 MB 9.3 MB/s eta 0:00:01
     ----------------- ---------------------- 3.1/7.1 MB 12.5 MB/s eta 0:00:01
     ------------------- -------------------- 3.4/7.1 MB 11.4 MB/s eta 0:00:01
     -------------------------- ------------- 4.8/7.1 MB 13.9 MB/s eta 0:00:01
     -------------------------------- ------- 5.7/7.1 MB 14.5 MB/s eta 0:00:01
     ------------------------------------- -- 6.6/7.1 MB 15.0 MB/s eta 0:00:01
     ---------------------------------------  


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else "N/A")
print("Allocated / Reserved (GB):", 
      round(torch.cuda.memory_allocated()/1e9, 2), 
      round(torch.cuda.memory_reserved()/1e9, 2))

CUDA available: False
GPU name: None
Total VRAM (GB): N/A
Allocated / Reserved (GB): 0.0 0.0


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "mistralai/Mistral-7B-v0.1"   # still large for pure CPU

# Safer choice for CPU:
# model_name = "microsoft/phi-2"          # ~2.7B
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,      # or torch.float16 if your CPU supports it well
    device_map="cpu",
    low_cpu_mem_usage=True,
)

print("Model loaded on CPU")

c:\Users\harpa\Documents\Others\PC\project_management\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:35<00:00, 17.82s/it]
c:\Users\harpa\Documents\Others\PC\project_management\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harpa\.cache\huggingface\hub\models--mistralai--Mistral-7B-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support sy

Model loaded on CPU


**Step 1: Load the base model quantised to four-bit precision, the "Q" in QLoRA.**

In [11]:
from llama_cpp import Llama

# ---------- Load the model ----------
llm = Llama(
    model_path="mistral-7b-v0.1.Q4_K_M.gguf",   # ← change to your actual filename/path
    n_ctx=2048,                                 # context length
    n_threads=8,                                # set to number of CPU cores (or leave default)
    n_batch=512,                                # helps speed a bit
    verbose=False,
)

print("✅ Model loaded successfully on CPU")

# ---------- Simple generation ----------
prompt = "Explain what Retrieval-Augmented Generation (RAG) is in simple terms."

output = llm(
    prompt,
    max_tokens=256,
    temperature=0.7,
    top_p=0.9,
    stop=["</s>", "User:", "Human:"],   # optional stop sequences
    echo=False,                         # do not repeat the prompt in the output
)

print("\n=== Model Response ===")
print(output["choices"][0]["text"].strip())

ValueError: Model path does not exist: mistral-7b-v0.1.Q4_K_M.gguf

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto",
)

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

**Step 2: Prepare the quantised model for training, then attach small, trainable LoRA adapter matrices.**

In [12]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

NameError: name 'base_model' is not defined

**Step 3: Prepare a small domain dataset, following this chapter's own guidance that even a modest example set is enough to start.**

In [ ]:
from datasets import Dataset

training_examples = [
    {"text": "### Instruction:\nSummarise this clause in plain English.\n### Clause:\nThe Lessee shall indemnify the Lessor against all claims.\n### Response:\nYou (the tenant) must cover the landlord's costs if someone makes a legal claim."},
    {"text": "### Instruction:\nSummarise this clause in plain English.\n### Clause:\nTermination for convenience may be exercised upon thirty days written notice.\n### Response:\nEither party can end the agreement for any reason, as long as they give thirty days' written notice first."},
    # a real fine-tuning run needs hundreds to thousands of examples in this
    # same instruction/response shape; two are shown here purely so this
    # snippet's structure is complete and copy-pasteable end to end
]

dataset = Dataset.from_list(training_examples)

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize)

**Verify the dataset shape before spending any GPU time**

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("gpt2")   # CPU-only stand-in for verifying dataset shape
tokenizer.pad_token = tokenizer.eos_token

dataset = Dataset.from_list(training_examples)

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize)
print(tokenized_dataset)
print("input_ids length:", len(tokenized_dataset[0]["input_ids"]))
print("decoded back:", repr(tokenizer.decode(tokenized_dataset[0]["input_ids"][:15])))

**Step 4: Run a short training loop with Hugging Face's own `Trainer`.**

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora-legal-adapter",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False, bf16=True,
    logging_steps=1,
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

model.save_pretrained("./lora-legal-adapter")

**Step 5: Load the adapter back onto the base model for inference, the part of this walkthrough every reader can actually run against any already-trained adapter.**

In [ ]:
from peft import PeftModel

inference_model = PeftModel.from_pretrained(base_model, "./lora-legal-adapter")

prompt = "### Instruction:\nSummarise this clause in plain English.\n### Clause:\nForce majeure events excuse performance during their continuance.\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)
output = inference_model.generate(**inputs, max_new_tokens=80)
print(tokenizer.decode(output[0], skip_special_tokens=True))

**How it works**

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

base = AutoModelForCausalLM.from_pretrained("hf-internal-testing/tiny-random-LlamaForCausalLM")

narrow = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
get_peft_model(base, narrow).print_trainable_parameters()

base2 = AutoModelForCausalLM.from_pretrained("hf-internal-testing/tiny-random-LlamaForCausalLM")

wide = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.05, bias="none")
get_peft_model(base2, wide).print_trainable_parameters()